# 🧠 DB-SLM: Instant Streaming Inference with Trained Model Weights

**Candidate:** Bilal Javed | Adept Tech Solutions

**Features:**
- ⚡ **Real-Time Token-by-Token Streaming** (`TextStreamer` prints responses live like ChatGPT)
- 🚀 **Fast KV-Caching Enabled** (`use_cache=True` runs generation in ~1–2 seconds on T4 GPU)
- 💾 **Google Drive Mounting** (Loads `db_slm_adapter` directly from Google Drive)

**Runtime:** Select `Runtime > Change runtime type > T4 GPU`

## Step 1: Mount Google Drive, Install Dependencies & Load Model

In [3]:
# 1. Mount Google Drive
from google.colab import drive
import os, zipfile
from pathlib import Path

print('Mounting Google Drive...')
drive.mount('/content/drive')

# 2. Install required inference packages
!pip install -q -U transformers peft bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer
from peft import PeftModel

BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# Check for adapter in Google Drive or local content
drive_zip = '/content/drive/MyDrive/db_slm_adapter.zip'
drive_folder = '/content/drive/MyDrive/db_slm_adapter'
local_dir = '/content/db_slm_adapter'

ADAPTER_DIR = local_dir
if Path(drive_folder).exists():
    ADAPTER_DIR = drive_folder
    print(f'✅ Found adapter folder in Google Drive: {drive_folder}')
elif Path(drive_zip).exists():
    print(f'Extracting adapter from Google Drive ({drive_zip})...')
    with zipfile.ZipFile(drive_zip, 'r') as zf:
        zf.extractall('/content/')
    ADAPTER_DIR = local_dir
elif Path('/content/db_slm_adapter.zip').exists():
    print('Extracting from uploaded zip in /content/...')
    !unzip -q /content/db_slm_adapter.zip -d /content/
    ADAPTER_DIR = local_dir

print(f'\nLoading base model: {BASE_MODEL} in 4-bit (T4 GPU)...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token if tokenizer.eos_token is not None else tokenizer.pad_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

# Attach trained LoRA adapter weights
if Path(ADAPTER_DIR).exists() and (Path(ADAPTER_DIR) / 'adapter_model.safetensors').exists():
    print(f'Attaching fine-tuned LoRA adapter from: {ADAPTER_DIR}...')
    model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
    print('✅ Fine-tuned LoRA weights successfully loaded!')
else:
    model = base_model
    print('⚠️ Adapter folder not found. Running base model without adapter.')

SYSTEM_PROMPT = """You are a specialized AI database assistant trained on two real enterprise databases.

DATABASE 1 — Online Retail (UK e-commerce, Dec 2010 – Dec 2011):
- Tables: customers (4,372 rows), products (1,124 rows), invoices (25,900 rows), invoice_items (54,873 rows).
- Key Facts: 38 countries. UK is the dominant domestic market (£685,023 confirmed revenue).
- Confirmed Revenue (is_cancelled=0): £1,276,568.60 (3,836 cancelled invoices / 14.8% cancellation rate excluded).
- Average Order Value: £100.52 per confirmed invoice.
- Top Countries by Revenue: United Kingdom (£685,023), Germany (£38,665), France (£35,492), EIRE (£22,848), Spain (£18,194).

DATABASE 2 — MIMIC-IV Clinical Demo (Beth Israel Deaconess Medical Center):
- Tables: patients (100 rows), admissions (275 rows), diagnoses_icd (4,506 rows), labevents (107,727 rows), prescriptions (18,087 rows), icustays (140 stays), chartevents (668,862 measurements).
- Key Facts: 100 patients (43 female, 57 male, mean age 61.8 years, 31 deceased).
- In-Hospital Mortality: 15 deaths across 275 admissions (5.5% mortality rate, hospital_expire_flag=1).
- Average ICU Length of Stay: 3.68 days across 140 stays.
- Lab Results: 107,727 total (37.4% flagged abnormal).
- seq_num=1 in diagnoses_icd indicates the primary diagnosis.

CROSS-DATABASE:
- patient_customer_bridge (100 links) maps retail customer_id directly to MIMIC subject_id.

IMPORTANT INSTRUCTION:
Provide direct, concise, and natural language conversational answers. Do NOT write SQL code, query syntax, or database calculation steps unless the user explicitly asks for SQL."""


# Set up Live Token-by-Token Streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

def ask_stream(query, max_new_tokens=300):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    print(f'❓ Question: {query}\n' + '='*70 + '\n🤖 Response:')
    with torch.no_grad():
        model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            use_cache=True,      # Fast KV-Cache generation (takes ~1-2s)
            pad_token_id=tokenizer.eos_token_id
        )
    print('='*70)

print('\n🚀 System is ready for fast live streaming queries!')

Mounting Google Drive...
Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 124.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 79.3 MB/s eta 0:00:00
Extracting adapter from Google Drive (/content/drive/MyDrive/db_slm_adapter.zip)...

Loading base model: Qwen/Qwen2.5-Coder-7B-Instruct in 4-bit (T4 GPU)...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Attaching fine-tuned LoRA adapter from: /content/db_slm_adapter...
✅ Fine-tuned LoRA weights successfully loaded!

🚀 System is ready for fast live streaming queries!


## Step 2: Query the Model (Live Streaming Output)

In [4]:
# ── Type any question below and run this cell ──────────────────────────────────
ask_stream('Which country generates the minimum revenue?')

❓ Question: Which country generates the minimum revenue?
🤖 Response:
The country that generates the minimum revenue is Spain with £18,194 confirmed.


### Suggested Questions to Try:
- `ask_stream('Explain the schema and table relationships of the Online Retail database in one line.')`
- `ask_stream('What is the difference between invoice_items and products tables?')`
- `ask_stream('What is the in-hospital mortality rate in MIMIC and how is it calculated?')`
- `ask_stream('Which country generates the highest retail revenue and why?')`
- `ask_stream('What does hospital_expire_flag mean in admissions?')`
- `ask_stream('How are retail customers and hospital patients linked in this system?')`

## Step 3: Automated Benchmark & Evaluation Suite

Runs an automated evaluation test suite measuring **Factual Exact Match (EM)** against ground-truth database facts and **Out-of-Distribution (OOD) Semantic Reasoning**.

In [6]:
def ask_eval(query, max_new_tokens=250):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# ── Benchmark Evaluation Test Suite (In-Domain + Out-of-Distribution) ──────────
BENCHMARK_SUITE = [
    # In-Domain Factual Accuracy (Ground-Truth SQL Verification)
    {
        'category': 'In-Domain Factual',
        'query': 'How many patients are in the MIMIC database?',
        'expected': ['100'],
        'gt_sql': 'SELECT COUNT(*) FROM patients -> 100'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'What is the confirmed revenue from the retail database?',
        'expected': ['1,276,568', '1276568', '1.27M', '1.28M'],
        'gt_sql': 'SELECT SUM(line_total) WHERE is_cancelled=0 -> £1,276,568.60'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'What is the in-hospital mortality rate in MIMIC?',
        'expected': ['5.5%', '5.5', '5.45%'],
        'gt_sql': 'SELECT AVG(hospital_expire_flag)*100 -> 5.45% (15/275)'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'How many unique customers are in the retail database?',
        'expected': ['4,372', '4372'],
        'gt_sql': 'SELECT COUNT(DISTINCT customer_id) FROM customers -> 4,372'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'What is the average ICU length of stay in MIMIC?',
        'expected': ['3.68'],
        'gt_sql': 'SELECT AVG(los) FROM icustays -> 3.68 days'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'How many hospital admissions are recorded in MIMIC?',
        'expected': ['275'],
        'gt_sql': 'SELECT COUNT(*) FROM admissions -> 275'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'What percentage of lab results in MIMIC are abnormal?',
        'expected': ['37.4%', '37.4'],
        'gt_sql': 'SELECT AVG(flag==abnormal)*100 -> 37.4% (40,275/107,727)'
    },
    {
        'category': 'In-Domain Factual',
        'query': 'How many patients in MIMIC have a recorded date of death?',
        'expected': ['31'],
        'gt_sql': 'SELECT COUNT(dod) FROM patients WHERE dod IS NOT NULL -> 31'
    },
    # Out-of-Distribution & Domain Reasoning
    {
        'category': 'OOD Reasoning',
        'query': 'Why is filtering is_cancelled=0 necessary when calculating financial totals?',
        'expected': ['cancelled', 'cancel', 'return', 'refund', '3,836', '14.8%'],
        'gt_sql': 'Excludes 3,836 cancelled/refunded invoices (14.8%)'
    },
    {
        'category': 'OOD Reasoning',
        'query': 'How are retail customers connected to MIMIC clinical patients?',
        'expected': ['patient_customer_bridge', 'bridge', 'subject_id', 'customer_id', '100'],
        'gt_sql': 'Linked via patient_customer_bridge table (100 links)'
    },
    {
        'category': 'OOD Reasoning',
        'query': 'Which country generates the top retail revenue and what is the amount?',
        'expected': ['United Kingdom', 'UK', '685,023'],
        'gt_sql': 'UK revenue -> £685,023'
    },
    {
        'category': 'OOD Reasoning',
        'query': 'What is the primary table that stores patient ICU stay details in MIMIC?',
        'expected': ['icustays'],
        'gt_sql': 'icustays table (140 rows)'
    }
]

print('=' * 80)
print('🧪 RUNNING AUTOMATED BENCHMARK EVALUATION (12 QUERIES)')
print('=' * 80)

passed_count = 0
in_domain_pass = 0
ood_pass = 0

for idx, item in enumerate(BENCHMARK_SUITE, 1):
    response = ask_eval(item['query'])
    has_match = any(exp.lower() in response.lower() for exp in item['expected'])
    status = '✅ PASS' if has_match else '❌ FAIL'

    if has_match:
        passed_count += 1
        if item['category'] == 'In-Domain Factual':
            in_domain_pass += 1
        else:
            ood_pass += 1

    print(f"\n[{idx}/12] [{item['category']}] {status}")
    print(f"  ❓ Query:    {item['query']}")
    print(f"  🎯 Expected: {item['expected']} (SQL: {item['gt_sql']})")
    print(f"  🤖 Model:    {response}")
    print('-' * 80)

total = len(BENCHMARK_SUITE)
print('\n' + '=' * 80)
print('📊 FINAL EVALUATION SCORECARD')
print('=' * 80)
print(f'Overall Accuracy:          {passed_count}/{total} ({passed_count/total*100:.1f}%)')
print(f'In-Domain Factual Match:   {in_domain_pass}/8 ({in_domain_pass/8*100:.1f}%)')
print(f'OOD Semantic Reasoning:    {ood_pass}/4 ({ood_pass/4*100:.1f}%)')
print('=' * 80)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🧪 RUNNING AUTOMATED BENCHMARK EVALUATION (12 QUERIES)

[1/12] [In-Domain Factual] ✅ PASS
  ❓ Query:    How many patients are in the MIMIC database?
  🎯 Expected: ['100'] (SQL: SELECT COUNT(*) FROM patients -> 100)
  🤖 Model:    There are 100 patients in the MIMIC-IV Clinical Demo database.
--------------------------------------------------------------------------------

[2/12] [In-Domain Factual] ✅ PASS
  ❓ Query:    What is the confirmed revenue from the retail database?
  🎯 Expected: ['1,276,568', '1276568', '1.27M', '1.28M'] (SQL: SELECT SUM(line_total) WHERE is_cancelled=0 -> £1,276,568.60)
  🤖 Model:    The confirmed revenue from the retail database is £1,276,568.60. This amount excludes 3,836 cancelled invoices, which represent an 14.8% cancellation rate.
--------------------------------------------------------------------------------

[3/12] [In-Domain Factual] ✅ PASS
  ❓ Query:    What is the in-hospital mortality rate in MIMIC?
  🎯 Expected: ['5.5%', '5.5', '5.45%'] (SQL: SELE